# W5: Artistic Telephone

First complete [Replicate basics](01_replicate_basics.ipynb).
Keep this notebook and `telephone_helpers.py` in the same folder.
Follow the [game guide](../notes/artistic_telephone.md).

Six groups of five play independently. Everyone starts a chain; each round you send to the
same next person **inside your group**. Describe the received image in words. It is not sent to the model.
Your prompt stays private until the reveal.

Begin in **practice** mode. Practice images are labelled test diagrams, not AI results.
Set up with `%pip install -r requirements.txt` if needed, then restart the kernel.


In [ ]:
import json
from pathlib import Path
from datetime import date
from IPython.display import display, Image, FileLink
import ipywidgets as widgets
from telephone_helpers import (
    MODELS, route, uploaded_image, practice_image, error_message,
    session_root, save_record, history, export_handoff, export_after_reveal,
)


## 1. Set your group and seat once

Choose the same session name throughout the class. The model dropdown starts with the existing
course example only. The instructor can curate `MODELS` in the helper file, checking each input schema.
Use the same selected model throughout the game. Each student uses their own Jupyter kernel and token.

**No widgets visible?** Restart after installation. As a fallback, replace the value assignments in the
next cell with plain values such as `group = "A"`, `seat = 1`, `round_number = 1`, `live = False`,
`model_label = next(iter(MODELS))`, and `session_name = "class-practice"`.
You can upload files through Jupyter's file browser instead of the upload widget.


In [ ]:
group_choice = widgets.Dropdown(options=list("ABCDEF"), description="Group:")
seat_choice = widgets.Dropdown(options=[1, 2, 3, 4, 5], description="Seat:")
round_choice = widgets.Dropdown(options=[1, 2, 3, 4, 5], description="Round:")
mode_choice = widgets.Dropdown(options=[("Offline practice", False), ("Live Replicate", True)], description="Mode:")
model_choice = widgets.Dropdown(options=list(MODELS), description="Model:", layout=widgets.Layout(width="650px"))
session_choice = widgets.Text(value=date.today().isoformat(), description="Session:")
display(widgets.VBox([group_choice, seat_choice, round_choice, mode_choice, model_choice, session_choice]))


## 2. Get this round's instructions

Rerun from this cell after changing the round. Do not rerun the widget creation cell, which resets choices.
Keep your group, seat, and model fixed. Starting a round clears the current preview/result references,
while saved history remains on disk.


In [ ]:
group, seat = group_choice.value, seat_choice.value
round_number, live = round_choice.value, mode_choice.value
model_label, session_name = model_choice.value, session_choice.value
assignment = route(group, seat, round_number)
root = session_root(session_name, group, seat)
incoming_bytes = None
source_filename = None
output = None
pending_context = None
image_bytes = None
saved_run = None
upload = None
print("Your shared folder:", assignment["folder"])
print("Receive:", assignment["incoming"] or "No image yet. Start from your group's theme.")
print("Export:", assignment["outgoing"])
print("Send inside your group to seat", assignment["send_to"])
print("Mode:", "LIVE (paid requests)" if live else "OFFLINE PRACTICE")


## 3. Inspect the received image (rounds 2 to 5)

Download only the assigned image from your group's SWITCHdrive folder, then upload it here.
Wait if it is not ready. Never use a different chain's image to catch up.
Do not share Replicate prediction pages because they expose prompts.


In [ ]:
if assignment["incoming"]:
    upload = widgets.FileUpload(accept=".png", multiple=False)
    print("Upload", assignment["incoming"], "then run the next cell.")
    display(upload)
else:
    print("Round 1: write a starter prompt from your theme.")


In [ ]:
# Optional file-browser fallback: upload in Jupyter, then enter that local path here.
incoming_path = ""
incoming_bytes = None
source_filename = None
if assignment["incoming"]:
    try:
        if incoming_path:
            path = Path(incoming_path)
            value = [{"name": path.name, "content": path.read_bytes()}]
        else:
            value = upload.value if upload is not None else []
        incoming_bytes = uploaded_image(value, assignment["incoming"])
        source_filename = assignment["incoming"]
        display(Image(data=incoming_bytes, width=480))
    except Exception:
        print("Load the exact assigned PNG from your group folder. Check its name and that it opens as an image, then rerun this cell.")
else:
    print("No incoming image is required for round 1.")


## 4. Describe it in your own words

Subject/action → setting → style → lighting → composition → mood.
In round 1, use your group's theme. Later, interpret only the received image.
Keep the model fixed. Do not copy another student's prompt.


In [ ]:
prompt = "At dawn, a cyclist crosses a foggy stone bridge, warm light, wide cinematic composition."


## 5. Connect privately (once per kernel, or when switching to live mode)

Paste the token only into the hidden input. Never share your notebook or prompt log during play.


In [ ]:
if live and ("client" not in globals() or client is None):
    import replicate
    from getpass import getpass
    token = getpass("Replicate API token (hidden): ").strip()
    if not token:
        raise ValueError("Enter a token or return to practice mode.")
    client = replicate.Client(api_token=token)
    del token
    print("Client ready.")
elif live:
    print("Using your existing client. Restart the kernel to change accounts.")
else:
    client = None
    print("Practice mode: no token needed.")


## 6. Generate one interpretation

The API call is the same as in the first notebook. A live request requires typing **GENERATE**.
Do not rerun it while waiting. If you interrupt a request, check Replicate before retrying.
The incoming image guides your writing but is **not** included in the API input.


In [ ]:
output = None
image_bytes = None
pending_context = None
saved_run = None
if not prompt.strip():
    print("Write a prompt first.")
elif assignment["incoming"] and incoming_bytes is None:
    print("Inspect the assigned incoming image before generating this round.")
elif live and client is None:
    print("Run the connection cell first.")
else:
    config = MODELS[model_label]
    settings = dict(config["input"])
    model_input = {**settings, "prompt": prompt}
    context = {**assignment, "prompt": prompt, "model": config["identifier"] if live else None,
               "settings": settings, "mode": "live" if live else "practice",
               "source_filename": source_filename, "observation": ""}
    if live:
        if input("One paid image request. Type GENERATE: ") == "GENERATE":
            try:
                output = client.run(config["identifier"], input=model_input, use_file_output=True)
                pending_context = context
                print("Result ready. Save below before changing the round.")
            except Exception as error:
                print(error_message(error))
    else:
        image_bytes = practice_image(round_number)
        pending_context = context
        print("Practice diagram ready; it is not a model interpretation of your prompt.")


## 7. Save your private record

Save before moving to the next round. This cell reads the returned file without requesting another image.
If a download fails, retry this cell. Prompt/settings are captured at generation time.
Re-running after a successful save reuses that saved result.


In [ ]:
observation = ""  # One short note: what persisted or changed?
if pending_context is None:
    print("No result to save yet.")
else:
    try:
        if saved_run is None:
            if image_bytes is None:
                image_bytes = output[0].read()
            pending_context["observation"] = observation
            saved_run = save_record(root, pending_context, image_bytes)
        display(Image(filename=str(saved_run / "image.png"), width=480))
        print("Saved privately:", saved_run)
    except Exception:
        print("Could not read or save this output. Check the returned image and folder permissions. Retry saving before requesting another paid generation.")


## 8. Export only the handoff image

Download the linked PNG and upload it to your group's SWITCHdrive folder, keeping its filename.
No prompt or notebook is included. The same filename in another group's folder belongs to a different chain.
Practice exports go in a separate local folder; use a separate practice folder on SWITCHdrive too.

If you made several attempts, select one before passing. The export step protects an already selected
image from accidental replacement. Ask the instructor if a passed image must be changed.


In [ ]:
if saved_run is None:
    print("Save a result first.")
else:
    try:
        handoff = export_handoff(root, saved_run)
        display(FileLink(str(handoff)))
        print("Upload this PNG only to", assignment["folder"], "on SWITCHdrive.")
        print("Next: wait for the round signal, change the Round dropdown, and rerun from step 2.")
    except FileExistsError as error:
        print(str(error))


## After the reveal: history, storyboard and reflection

Do not project or share your private history until the instructor reveals the prompts.
Each row represents your own contribution, not the whole group's chain.
Use saved images for comparison without new API calls.


In [ ]:
records = history(root)
for item in records:
    print("Round", item["round"], "|", item["outgoing"], "|", item["mode"])
    print("Prompt:", item["prompt"])
    print("Observation:", item["observation"])
    display(Image(filename=str(Path(item["run_path"]) / "image.png"), width=240))


Use the fields below to plan three shots and one repair. Sketch on paper if preferred and describe it here.
An additional generation is optional. Use the basics notebook for that refinement so the game's passed files stay fixed.


In [ ]:
reflection = {
    "what_persisted": "",
    "what_drifted": "",
    "consistency_details": "",
    "storyboard": [
        {"shot": 1, "scene": "", "composition": ""},
        {"shot": 2, "scene": "", "composition": ""},
        {"shot": 3, "scene": "", "composition": ""},
    ],
    "revised_prompt": "",
    "reason_for_change": "",
    "what_to_check_next": "",
}
root.mkdir(parents=True, exist_ok=True)
(root / "reflection.json").write_text(json.dumps(reflection, indent=2), encoding="utf-8")
print("Reflection saved. Fill in the fields and rerun this cell when ready.")


## Final submission, only after the reveal

This archive includes your prompts, images and reflection. It must not be used for image passing.
It excludes your notebook, API client and credentials. Submit through the instructor's class folder.


In [ ]:
AFTER_REVEAL = False  # Change to True only when the instructor opens the reveal.
if AFTER_REVEAL:
    archive = export_after_reveal(root)
    display(FileLink(str(archive)))
else:
    print("Prompt archive remains private until the reveal.")
